# Narzędzia do interaktywnej eksploracji danych

**Eksploracja danych · laboratorium**

Dzisiaj zbudujemy mały panel analityczny do danych sprzedażowych zapisanych w bazie SQLite. Przejdziemy trzy kroki:

| Część | Temat | Co powstaje |
|---|---|---|
| A | SQLite i pandas | funkcje pobierające i agregujące dane |
| B | DuckDB | te same agregacje w SQL na DataFrame i pliku Parquet |
| C | Plotly Express | interaktywne wykresy i mapa |
| D | ipywidgets | panel z filtrami i formularzem w notebooku |

Na końcu prowadzący pokaże, jak **te same funkcje** zamienić w aplikację (Streamlit, Panel, Mercury).

**Zaliczenie:** uzupełniony notebook w Twoim repozytorium na GitHubie. Komórki oznaczone `# TODO` są do uzupełnienia, a komórki `assert` po nich sprawdzają, czy wynik jest poprawny.

## 0. Przygotowanie środowiska

In [1]:
# W Colabie większość bibliotek jest już zainstalowana; upewniamy się co do wersji
%pip install -q "plotly>=5.24" duckdb

Note: you may need to restart the kernel to use updated packages.


In [3]:
import sqlite3
import pandas as pd
import plotly.express as px
from IPython.display import display
import ipywidgets as widgets

pd.set_option("display.max_columns", 20)

Poniższa komórka tworzy plik `sales.db` z dwiema tabelami: `stores` (sklepy) i `sales` (transakcje).
Uruchom ją, nie musisz jej modyfikować. Można ją uruchamiać wielokrotnie: baza jest za każdym razem tworzona od nowa.

In [1]:
"""Generator bazy sales.db do zajęć z eksploracji danych.

Tworzy dwie tabele:
  stores(store_id, city, latitude, longitude)
  sales(id, product, quantity, price, date, store_id)

Dane są syntetyczne (ziarno losowania stałe), ok. 2500 transakcji
z okresu 1.09-30.11.2026: weekendy są słabsze, jest lekki trend wzrostowy
i Black Friday (27.11). W danych ukryto kilka błędów do znalezienia.
"""
import sqlite3
from datetime import date, timedelta

import numpy as np

STORES = [
    (1, "Łódź", 51.7592, 19.4560),
    (2, "Warszawa", 52.2297, 21.0122),
    (3, "Kraków", 50.0647, 19.9450),
    (4, "Gdańsk", 54.3520, 18.6466),
    (5, "Wrocław", 51.1079, 17.0385),
    (6, "Poznań", 52.4064, 16.9252),
]
STORE_WEIGHTS = [0.15, 0.30, 0.18, 0.10, 0.15, 0.12]

# produkt: (cena bazowa, średnia liczba sztuk w transakcji)
PRODUCTS = {
    "Laptop": (3200, 1.3),
    "Monitor": (850, 2.0),
    "Klawiatura": (150, 3.0),
    "Mysz": (55, 4.0),
    "Słuchawki": (300, 2.0),
    "Kamera": (250, 1.5),
}


def create_sales_db(path="sales.db", n_days=91, seed=42):
    rng = np.random.default_rng(seed)
    conn = sqlite3.connect(path)
    cur = conn.cursor()
    # DROP zamiast CREATE IF NOT EXISTS: ponowne uruchomienie nie dubluje danych
    cur.executescript("""
        DROP TABLE IF EXISTS sales;
        DROP TABLE IF EXISTS stores;
        CREATE TABLE stores (
            store_id  INTEGER PRIMARY KEY,
            city      TEXT NOT NULL,
            latitude  REAL NOT NULL,
            longitude REAL NOT NULL
        );
        CREATE TABLE sales (
            id       INTEGER PRIMARY KEY AUTOINCREMENT,
            product  TEXT NOT NULL,
            quantity INTEGER NOT NULL,
            price    REAL NOT NULL,
            date     TEXT NOT NULL,
            store_id INTEGER NOT NULL REFERENCES stores(store_id)
        );
    """)
    cur.executemany("INSERT INTO stores VALUES (?, ?, ?, ?)", STORES)

    rows = []
    start = date(2026, 9, 1)
    names = list(PRODUCTS)
    for d in range(n_days):
        day = start + timedelta(days=d)
        factor = 1 + 0.4 * d / n_days               # trend
        if day.weekday() >= 5:                      # weekend
            factor *= 0.6
        if day == date(2026, 11, 27):               # Black Friday
            factor *= 3.0
        for _ in range(rng.poisson(26 * factor)):
            product = rng.choice(names)
            base_price, mean_qty = PRODUCTS[product]
            qty = int(rng.poisson(mean_qty - 1) + 1)
            price = round(base_price * rng.uniform(0.9, 1.1), 2)
            if day == date(2026, 11, 27):
                price = round(price * 0.8, 2)       # promocja
            store = int(rng.choice([s[0] for s in STORES], p=STORE_WEIGHTS))
            rows.append((product, qty, price, day.isoformat(), store))

    # ukryte błędy danych do znalezienia podczas eksploracji
    rows[150] = ("Laptop", 1, 32000.0, rows[150][3], rows[150][4])   # literówka w cenie
    rows[900] = ("Mysz", -5, 52.0, rows[900][3], rows[900][4])       # ujemna ilość
    rows[1700] = ("Monitor", 400, 860.0, rows[1700][3], rows[1700][4])  # podejrzanie duże zamówienie

    cur.executemany(
        "INSERT INTO sales (product, quantity, price, date, store_id) VALUES (?, ?, ?, ?, ?)",
        rows,
    )
    conn.commit()
    conn.close()
    return len(rows)

n = create_sales_db("sales.db")
print(f"Utworzono sales.db: {n} transakcji")

Utworzono sales.db: 2451 transakcji


## A. SQLite i pandas

### A.0 Rozpoznanie bazy
Zanim cokolwiek policzymy, sprawdzamy, **co jest w bazie**: jakie tabele, jakie kolumny, ile rekordów.

In [4]:
conn = sqlite3.connect("sales.db")

# lista tabel zapisana w systemowej tabeli sqlite_master
pd.read_sql("SELECT name, sql FROM sqlite_master WHERE type = 'table'", conn)

,name,sql
0,sqlite_sequence,"CREATE TABLE sqlite_sequence(name,seq)"
1,stores,CREATE TABLE stores (\n store_id I...
2,sales,CREATE TABLE sales (\n id INT...


In [5]:
# klasyczne API: kursor + fetchall() zwraca listę krotek
cur = conn.cursor()
cur.execute("SELECT * FROM sales LIMIT 5")
for row in cur.fetchall():
    print(row)

(1, 'Klawiatura', 2, 164.27, '2026-09-01', 5)
(2, 'Kamera', 2, 247.52, '2026-09-01', 2)
(3, 'Monitor', 3, 803.63, '2026-09-01', 3)
(4, 'Kamera', 1, 266.38, '2026-09-01', 4)
(5, 'Laptop', 1, 3501.25, '2026-09-01', 6)


In [6]:
# wygodniej: od razu DataFrame
pd.read_sql("SELECT * FROM stores", conn)

,store_id,city,latitude,longitude
0,1,Łódź,51.7592,19.4560
1,2,Warszawa,52.2297,21.0122
2,3,Kraków,50.0647,19.9450
3,4,Gdańsk,54.3520,18.6466
4,5,Wrocław,51.1079,17.0385
5,6,Poznań,52.4064,16.9252


### A.1 Agregacja: SQL czy pandas?
To samo pytanie, *„ile zarobił każdy produkt?”*, można zadać bazie albo pandasowi. Wynik jest ten sam.

In [7]:
sql_result = pd.read_sql("""
    SELECT product, SUM(quantity * price) AS revenue
    FROM sales
    GROUP BY product
    ORDER BY revenue DESC
""", conn)

raw = pd.read_sql("SELECT * FROM sales", conn)
pandas_result = (raw.assign(revenue=raw.quantity * raw.price)
                    .groupby("product", as_index=False)["revenue"].sum()
                    .sort_values("revenue", ascending=False))

display(sql_result, pandas_result)

,product,revenue
0,Laptop,1866941.32
1,Monitor,1041825.66
2,Słuchawki,244746.23
3,Klawiatura,170774.34
4,Kamera,155176.99
5,Mysz,89181.60


,product,revenue
2,Laptop,1866941.32
3,Monitor,1041825.66
5,Słuchawki,244746.23
1,Klawiatura,170774.34
0,Kamera,155176.99
4,Mysz,89181.60


> **Kiedy co?** Przy dużych danych lepiej, żeby filtrowanie i agregację wykonała baza, bo do Pythona trafia wtedy tylko wynik. Przy małych danych i eksploracji pandas jest wygodniejszy.

### A.2 Zadanie: JOIN
Tabela `sales` ma tylko `store_id`. Miasto i współrzędne są w tabeli `stores`.
Napisz zapytanie, które zwróci **przychód i liczbę transakcji w każdym mieście** (kolumny: `city`, `revenue`, `transactions`), posortowane malejąco po przychodzie.

In [ ]:
# TODO: uzupełnij zapytanie (JOIN + GROUP BY)
query = """
    SELECT
        st.city as city,
        sum(sa.quantity * sa.price) as revenue,
        count(*) as transactions
    from sales as sa
    inner join stores as st on sa.store_id = st.store_id
    group by st.city
    order by revenue DESC;
"""
by_city = pd.read_sql(query, conn)
by_city

In [ ]:
assert list(by_city.columns) == ["city", "revenue", "transactions"]
assert by_city.iloc[0]["city"] == "Warszawa"
assert by_city["transactions"].sum() == pd.read_sql("SELECT COUNT(*) AS n FROM sales", conn)["n"][0]
print("OK")

### A.3 Zapytania z parametrami
Wartości od użytkownika (np. z widżetu) **nigdy** nie wklejamy do SQL f-stringiem, bo grozi to *SQL injection* i błędami przy apostrofach.
Zamiast tego w zapytaniu stawiamy `?`, a wartości przekazujemy osobno w `params`.

In [ ]:
product = "Laptop"

# ŹLE:   f"SELECT * FROM sales WHERE product = '{product}'"
# DOBRZE:
pd.read_sql("SELECT * FROM sales WHERE product = ? AND date >= ?", conn,
            params=[product, "2026-11-01"]).head()

### A.4 Zadanie: funkcje warstwy logiki
Te trzy funkcje będą używane do końca zajęć: przez wykresy, widżety, a na koniec przez aplikacje.

1. `get_sales(conn, products=None, date_from=None, date_to=None)` zwraca DataFrame z kolumnami
   `id, product, quantity, price, revenue, date, city, latitude, longitude`, gdzie `revenue = quantity * price`.
   Filtry są opcjonalne, gdy podano `None`, nie filtrujemy. Użyj parametrów `?`. Kolumnę `date` zamień na `datetime`.
2. `daily_revenue(df)` zwraca kolumny `date, revenue` (suma przychodu w każdym dniu).
3. `revenue_by_product(df)` zwraca kolumny `product, quantity, revenue` posortowane malejąco po `revenue`.

*Wskazówka do filtra listy produktów:* `"IN (" + ", ".join("?" * len(products)) + ")"` daje `IN (?, ?, ?)`.

In [ ]:
def get_sales(conn, products=None, date_from=None, date_to=None):
    query = """
        SELECT s.id, s.product, s.quantity, s.price,
               s.quantity * s.price AS revenue,
               s.date, st.city, st.latitude, st.longitude
        FROM sales AS s
        JOIN stores AS st ON s.store_id = st.store_id
        WHERE 1 = 1
    """
    params = []
    # TODO: dopisz warunki dla products, date_from, date_to (z parametrami ?)
    if products is not None:
        query += " AND s.product IN (" + ", ".join("?" * len(products)) + ")"
        for p in products:
            params.append(p)
    if date_from is not None:
        query += " AND s.date >= ?"
        params.append(str(date_from))
    if date_to is not None:
        query += " AND s.date <= ?"
        params.append(str(date_to))

    df = pd.read_sql(query + " ORDER BY s.date", conn, params=params)
    # TODO: zamień kolumnę date na datetime
    df["date"] = pd.to_datetime(df["date"])
    return df


def daily_revenue(df):
    # TODO
    wynik = df.groupby("date", as_index=False)["revenue"].sum()
    return wynik


def revenue_by_product(df):
    # TODO
    wynik = df.groupby("product", as_index=False)[["quantity", "revenue"]].sum()
    wynik = wynik.sort_values("revenue", ascending=False)
    return wynik

In [ ]:
df = get_sales(conn)
assert len(df) == pd.read_sql("SELECT COUNT(*) AS n FROM sales", conn)["n"][0]
assert pd.api.types.is_datetime64_any_dtype(df["date"])

part = get_sales(conn, products=["Mysz", "Kamera"], date_from="2026-10-01", date_to="2026-10-31")
assert set(part["product"]) == {"Mysz", "Kamera"}
assert part["date"].min() >= pd.Timestamp("2026-10-01") and part["date"].max() <= pd.Timestamp("2026-10-31")

assert list(daily_revenue(df).columns) == ["date", "revenue"]
assert list(revenue_by_product(df).columns) == ["product", "quantity", "revenue"]
assert revenue_by_product(df).iloc[0]["product"] == "Laptop"
print("OK")

### A.5 Zadanie: czy dane są czyste?
W bazie ukryto **trzy** błędne lub podejrzane rekordy. Znajdź je (np. `describe()`, sortowanie, warunki logiczne) i krótko opisz w komórce Markdown, co jest z nimi nie tak.
Na razie ich **nie usuwaj**: zobaczysz, jak wyglądają na wykresach.

In [ ]:
# TODO: znajdź podejrzane rekordy
print("statystyki:", df.describe())

# ujemna albo zerowa ilość
print("ilość <= 0:", df[df["quantity"] <= 0])

# największe ilości
print("największe ilości:", df.sort_values("quantity", ascending=False).head())

# ceny - porównanie z medianą ceny produktu
mediana_ceny = df.groupby("product")["price"].transform("median")
print("cena dużo większa niż zwykle:", df[df["price"] > 2 * mediana_ceny])

*Twoje wnioski:* …  
1. Laptop za 32000 zł - zwykle laptop kosztuje ok. 3200 zł, więc to pewnie literówka (dodatkowe 0).  
2. Mysz z ilością -5 - ilość nie może być ujemna, może to zwrot ale wpisany jako sprzedaż.  
3. 400 monitorów w jednej transakcji - zwykle są 1-3 sztuki, więc to podejrzanie duże zamówienie (albo błąd, albo zamówienie hurtowe).

## B. DuckDB: SQL bez serwera, prosto na danych

[DuckDB](https://duckdb.org) to analityczna (kolumnowa) baza danych działająca, podobnie jak SQLite, w procesie Pythona.
Różnica: SQLite jest zoptymalizowany pod wiele małych zapisów (aplikacje), a DuckDB pod szybkie agregacje dużych zbiorów (analiza).
DuckDB umie wykonać SQL **bezpośrednio na DataFrame**, na plikach CSV i Parquet, bez wczytywania ich do bazy.

In [ ]:
import duckdb

# SQL na DataFrame `df`: DuckDB znajduje zmienną po nazwie
duckdb.sql("""
    SELECT city, product, ROUND(SUM(revenue)) AS revenue
    FROM df
    GROUP BY city, product
    ORDER BY revenue DESC
    LIMIT 5
""").df()

In [ ]:
# zapis do Parquet i zapytanie bezpośrednio na pliku
df.to_parquet("sales.parquet", index=False)

duckdb.sql("""
    SELECT date_trunc('week', date) AS week, SUM(revenue) AS revenue
    FROM 'sales.parquet'
    GROUP BY week
    ORDER BY week
""").df().head()

### B.1 Zadanie
Używając DuckDB na pliku `sales.parquet`, policz dla każdego **dnia tygodnia** średni dzienny przychód.
Czy weekendy różnią się od dni roboczych? Który dzień wypada podejrzanie wysoko i dlaczego (wróć do A.5)?

*Wskazówki:* `dayname(date)` lub `isodow(date)`; najpierw zsumuj przychód w dniu (podzapytanie / `WITH`), potem uśrednij.

In [ ]:
# TODO
weekday = duckdb.sql("""
    WITH dzienne AS (
        SELECT date, SUM(revenue) AS revenue
        FROM 'sales.parquet'
        GROUP BY date
    )
    SELECT isodow(date) AS nr, dayname(date) AS day, AVG(revenue) AS avg_revenue
    FROM dzienne
    GROUP BY nr, day
    ORDER BY nr
""").df()
weekday

*Odpowiedź:* …  
Weekendy są dużo słabsze (sobota ok. 25 tys., niedziela ok. 22 tys.), dni robocze to ok. 35-45 tys. zł.  
Podejrzanie wysoko wypada poniedziałek (ok. 66 tys.), bo 9.11 był poniedziałek i wtedy jest rekord z 400 monitorami (344 tys. zł). Bez niego poniedziałek miałby ok. 39 tys., czyli tyle co inne dni.

## C. Plotly Express: interaktywne wykresy

Pierwszy poziom interaktywności nie wymaga żadnego kodu: wykres Plotly ma tooltip po najechaniu, zoom, przesuwanie i klikalną legendę.
Składnia `px.<typ>(df, x=..., y=..., color=...)` przypomina seaborn.

In [ ]:
fig = px.line(daily_revenue(df), x="date", y="revenue", markers=True,
              title="Sprzedaż dzienna [zł]",
              labels={"date": "Data", "revenue": "Przychód [zł]"})
fig.show()

Najedź na największy pik i powiększ go myszą. Który rekord z zadania A.5 go powoduje?

*Odpowiedź:* …  
Największy pik jest 9.11.2026 (ok. 408 tys. zł) - to rekord z 400 monitorami w Warszawie.

### C.1 Zadanie: wykres słupkowy
Narysuj liczbę **sprzedanych sztuk** wg produktu (`revenue_by_product`). Dodaj przychód do tooltipa (`hover_data`) i polskie podpisy osi (`labels`).

In [ ]:
# TODO
fig = px.bar(revenue_by_product(df), x="product", y="quantity", hover_data=["revenue"],
             title="Sprzedane sztuki wg produktu",
             labels={"product": "Produkt", "quantity": "Sprzedane sztuki", "revenue": "Przychód [zł]"})
fig.show()

### C.2 Zadanie: mapa sklepów
Zagreguj dane po `city, latitude, longitude` (suma `revenue`, liczba transakcji) i pokaż sklepy na mapie za pomocą `px.scatter_map`.
Wielkość punktu ma odpowiadać przychodowi.

In [ ]:
def revenue_by_city(df):
    # TODO: kolumny city, latitude, longitude, revenue, transactions
    wynik = df.groupby(["city", "latitude", "longitude"], as_index=False).agg(
        revenue=("revenue", "sum"),
        transactions=("id", "count")
    )
    return wynik

# TODO: px.scatter_map(..., lat=..., lon=..., size=..., hover_name=..., zoom=5)
fig = px.scatter_map(revenue_by_city(df), lat="latitude", lon="longitude", size="revenue",
                     hover_name="city", hover_data=["transactions"], zoom=5,
                     title="Przychód wg sklepu")
fig.show()

### C.3 Zadanie (krótkie): jeden wykres, więcej informacji
Narysuj sprzedaż dzienną **osobno dla każdego produktu** (jedna linia na produkt, `color="product"`).
Kliknij dwukrotnie produkt w legendzie: co się dzieje?

In [ ]:
# TODO
dzienne_produkty = df.groupby(["date", "product"], as_index=False)["revenue"].sum()

fig = px.line(dzienne_produkty, x="date", y="revenue", color="product",
              title="Sprzedaż dzienna wg produktu [zł]",
              labels={"date": "Data", "revenue": "Przychód [zł]", "product": "Produkt"})
fig.show()

*Odpowiedź:* …  
Po dwukrotnym kliknięciu na produkt w legendzie zostaje tylko jego linia, a reszta się chowa. Kolejne dwukrotne kliknięcie pokazuje z powrotem wszystkie produkty.

## D. ipywidgets: panel w notebooku

Drugi poziom: **własne kontrolki**. Biblioteka `ipywidgets` daje suwaki, listy, przyciski i pola tekstowe, które działają w Colabie i Jupyterze.

### D.0 Najszybciej: `interact`
`interact` sam dobiera widżet do typu argumentu: lista daje rozwijaną listę, a krotka liczb suwak.

In [ ]:
@widgets.interact(product=get_sales(conn)["product"].unique().tolist(), min_qty=(1, 10))
def show(product, min_qty):
    sub = df[(df["product"] == product) & (df["quantity"] >= min_qty)]
    print(f"{len(sub)} transakcji, przychód {sub['revenue'].sum():,.0f} zł")
    px.histogram(sub, x="price", nbins=30, title=f"Rozkład cen: {product}").show()

### D.1 Budowa panelu krok po kroku
Pełny panel składa się z trzech elementów:
1. **widżety**, które przechowują stan (`.value`),
2. **funkcja `update`**, która czyta `.value`, wywołuje funkcje z części A i C i rysuje wynik w `Output`,
3. **połączenie**: `widget.observe(update, names="value")` wywołuje `update` przy każdej zmianie.

In [ ]:
products = sorted(df["product"].unique())

w_products = widgets.SelectMultiple(options=products, value=products, description="Produkty", rows=6)
w_from = widgets.DatePicker(description="Od", value=pd.Timestamp("2026-09-01").date())
w_to = widgets.DatePicker(description="Do", value=pd.Timestamp("2026-11-30").date())
out = widgets.Output()

widgets.VBox([w_products, w_from, w_to])   # podgląd samych kontrolek

### D.2 Zadanie: funkcja `update`
Uzupełnij `update`, tak aby po każdej zmianie filtra:
- pobrała dane przez `get_sales(...)` z wartościami widżetów,
- wypisała liczbę transakcji i łączny przychód,
- narysowała wykres dzienny, słupkowy i mapę (możesz skopiować kod z części C).

Następnie podłącz `update` do wszystkich trzech widżetów.

In [ ]:
def update(change=None):
    with out:
        out.clear_output(wait=True)
        # TODO: data = get_sales(conn, products=..., date_from=..., date_to=...)
        data = get_sales(conn, products=list(w_products.value), date_from=w_from.value, date_to=w_to.value)

        # TODO: print podsumowania
        print("transakcje:", len(data))
        print("przychód:", round(data["revenue"].sum(), 2), "zł")

        if len(data) == 0:
            print("brak danych dla tych filtrów")
            return

        # TODO: trzy wykresy (.show())
        px.line(daily_revenue(data), x="date", y="revenue", markers=True,
                title="Sprzedaż dzienna [zł]",
                labels={"date": "Data", "revenue": "Przychód [zł]"}).show()
        px.bar(revenue_by_product(data), x="product", y="quantity", hover_data=["revenue"],
               title="Sprzedane sztuki wg produktu",
               labels={"product": "Produkt", "quantity": "Sprzedane sztuki", "revenue": "Przychód [zł]"}).show()
        px.scatter_map(revenue_by_city(data), lat="latitude", lon="longitude", size="revenue",
                       hover_name="city", hover_data=["transactions"], zoom=5,
                       title="Przychód wg sklepu").show()


# TODO: podłącz update do widżetów (observe)
w_products.observe(update, names="value")
w_from.observe(update, names="value")
w_to.observe(update, names="value")

update()
dashboard = widgets.HBox([widgets.VBox([w_products, w_from, w_to]), out])
display(dashboard)

### D.3 Zadanie: formularz dodawania sprzedaży
Panel eksploracyjny często pozwala też **poprawić lub dopisać dane**. Zbuduj formularz:
`Dropdown` (produkt), `IntText` (ilość), `FloatText` (cena), `DatePicker` (data), `Dropdown` (sklep: etykieta miasta, wartość `store_id`) i `Button` „Dodaj”.

Po kliknięciu:
- sprawdź, czy ilość i cena są dodatnie (jeśli nie, wypisz komunikat i nic nie zapisuj),
- wstaw rekord zapytaniem `INSERT` z parametrami `?` i zrób `conn.commit()`,
- wywołaj `update()`, żeby panel z D.2 się odświeżył.

*Wskazówka:* `Dropdown(options=[("Łódź", 1), ("Warszawa", 2), ...])` pokazuje nazwę, a w `.value` jest liczba.

In [ ]:
stores = pd.read_sql("SELECT store_id, city FROM stores", conn)

# TODO: widżety formularza
sklepy = []
for i in range(len(stores)):
    sklepy.append((stores["city"][i], int(stores["store_id"][i])))

f_product = widgets.Dropdown(options=products, description="Produkt")
f_quantity = widgets.IntText(value=1, description="Ilość")
f_price = widgets.FloatText(value=100.0, description="Cena")
f_date = widgets.DatePicker(description="Data", value=pd.Timestamp("2026-11-30").date())
f_store = widgets.Dropdown(options=sklepy, description="Sklep")
f_button = widgets.Button(description="Dodaj", button_style="success")
f_msg = widgets.Output()


def add_sale(_):
    with f_msg:
        f_msg.clear_output()
        # TODO: walidacja, INSERT z parametrami, commit, update()
        if f_quantity.value <= 0 or f_price.value <= 0:
            print("Ilość i cena muszą być dodatnie, nic nie zapisano.")
            return
        if f_date.value is None:
            print("Wybierz datę, nic nie zapisano.")
            return

        conn.execute(
            "INSERT INTO sales (product, quantity, price, date, store_id) VALUES (?, ?, ?, ?, ?)",
            [f_product.value, f_quantity.value, f_price.value, str(f_date.value), f_store.value]
        )
        conn.commit()
        print("Dodano:", f_product.value, f_quantity.value, "szt.", f_price.value, "zł")
        update()


# TODO: f_button.on_click(add_sale)
f_button.on_click(add_sale)
# TODO: display(widgets.VBox([...]))
display(widgets.VBox([f_product, f_quantity, f_price, f_date, f_store, f_button, f_msg]))

## E. Co dalej: od notebooka do aplikacji

Panel z części D działa tylko dla kogoś, kto uruchomi notebook. Żeby udostępnić go osobie, która nie zna Pythona, potrzebujemy **aplikacji**.
Ponieważ logika (`get_sales`, `daily_revenue`, …) jest oddzielona od widżetów, zmienia się tylko warstwa interfejsu:

| Narzędzie | Wejście | Uruchomienie |
|---|---|---|
| **Streamlit** | skrypt `app.py` | `streamlit run app.py` |
| **Panel** | skrypt lub notebook | `panel serve app.py` |
| **Mercury** | notebook z widżetami `mr.*` | `mercury` (w katalogu z notebookiem) |

Gotowe przykłady pokazane na zajęciach: `app_streamlit.py`, `app_panel.py`, `demo_mercury.ipynb` (wymagają pliku `sales_logic.py` z tymi samymi funkcjami).
To dobry punkt startu, jeśli projekt na wyższą ocenę ma mieć formę interaktywnej aplikacji.

### Na koniec
- [ ] wszystkie komórki `assert` przechodzą,
- [ ] w A.5 opisane są trzy podejrzane rekordy,
- [ ] notebook jest zapisany w Twoim repozytorium na GitHubie.

In [ ]:
conn.close()